In [6]:
import pandas as pd

# 读取数据
df = pd.read_csv("合并23最终数据编码.csv")  # 或你已有的 DataFrame

# 映射温度区间标签
def temp_category(temp):
    if temp > 30:
        return 'above30'
    elif 20 < temp <= 30:
        return '30-20'
    elif 10 < temp <= 20:
        return '20-10'
    elif 0 <= temp <= 10:
        return '10-0'
    else:
        return 'below0'

df['temp_range'] = df['气温'].apply(temp_category)

# 时间段标签
df['time_label'] = df['时间段'].map({0: 'day', 1: 'night'})

# 定义天气类型列和映射名称
weather_columns = {
    '天气_多云': 'cloudy',
    '天气_晴': 'sunny',
    '天气_阴': 'overcast',
    '天气_雨': 'rainy',
    '天气_雪': 'snow',
    '天气_雾霾沙尘': 'haze'
}

# 初始化结果字典
weather_stats = {'day': {}, 'night': {}}
temp_bins = ['above30', '30-20', '20-10', '10-0', 'below0']
weather_types = list(weather_columns.values())

for time_label in ['day', 'night']:
    for temp_range in temp_bins:
        # 筛选当前组的数据
        group_df = df[(df['time_label'] == time_label) & (df['temp_range'] == temp_range)]
        # 初始化天气计数字典
        weather_count = {}
        for col, new_name in weather_columns.items():
            weather_count[new_name] = int(group_df[col].sum())
        # 存入主字典
        weather_stats[time_label][temp_range] = weather_count

# 显示结构
import pprint
pprint.pprint(weather_stats, sort_dicts=False)
#输出为.json文件
import json
with open('weather_stats.json', 'w', encoding='utf-8') as f:
    json.dump(weather_stats, f, ensure_ascii=False, indent=4)

{'day': {'above30': {'cloudy': 123,
                     'sunny': 56,
                     'overcast': 37,
                     'rainy': 24,
                     'snow': 0,
                     'haze': 0},
         '30-20': {'cloudy': 99,
                   'sunny': 69,
                   'overcast': 52,
                   'rainy': 93,
                   'snow': 0,
                   'haze': 0},
         '20-10': {'cloudy': 69,
                   'sunny': 51,
                   'overcast': 34,
                   'rainy': 44,
                   'snow': 0,
                   'haze': 0},
         '10-0': {'cloudy': 64,
                  'sunny': 49,
                  'overcast': 64,
                  'rainy': 17,
                  'snow': 19,
                  'haze': 1},
         'below0': {'cloudy': 0,
                    'sunny': 1,
                    'overcast': 1,
                    'rainy': 0,
                    'snow': 0,
                    'haze': 0}},
 'night': {'above30': {'

In [2]:
import pandas as pd

# 读取CSV文件
df = pd.read_csv('end.csv')

# 定义温度区间函数
def get_temp_group(temp):
    if temp > 30:
        return 'above30'
    elif 20 < temp <= 30:
        return '30-20'
    elif 10 < temp <= 20:
        return '20-10'
    elif 0 <= temp <= 10:
        return '10-0'
    else:
        return 'below0'

# 添加温度区间列
df['temp_group'] = df['气温'].apply(get_temp_group)

# 初始化最终的统计数据字典
pollution_stats_calculated = {
    "day": {},
    "night": {}
}

# 按时间段划分
for period, period_name in enumerate(['day', 'night']):
    # 筛选时间段
    period_df = df[df['时间段'] == period]
    
    # 按温度区间分组
    grouped = period_df.groupby('temp_group')
    
    # 计算每组的平均值
    stats = {}
    for group, group_df in grouped:
        stats[group] = group_df[['SO2', 'NO2', 'PM10', 'CO', 'O31小时', 'O38小时', 'PM2.5']].mean().round(2).tolist()
    
    pollution_stats_calculated[period_name] = stats

# 输出结果
import pprint
pprint.pprint(pollution_stats_calculated)
# 输出为JSON文件
import json
with open('pollution_stats_calculated.json', 'w', encoding='utf-8') as f:
    json.dump(pollution_stats_calculated, f, ensure_ascii=False, indent=4)

{'day': {'10-0': [62.37, 42.69, 114.46, 78.48, 21.8, 20.65, 234.57],
         '20-10': [34.74, 33.84, 101.25, 65.86, 21.76, 27.79, 173.16],
         '30-20': [24.33, 27.8, 83.32, 61.85, 31.66, 39.36, 150.13],
         'above30': [23.22, 21.74, 71.06, 49.61, 43.61, 58.02, 90.15],
         'below0': [39.27, 32.49, 95.45, 82.79, 22.88, 27.3, 245.59]},
 'night': {'10-0': [40.38, 38.21, 108.53, 62.14, 25.15, 32.07, 163.44],
           '20-10': [22.74, 24.68, 77.63, 57.36, 37.18, 47.24, 125.54],
           '30-20': [21.81, 18.81, 68.65, nan, nan, nan, nan],
           'below0': [47.59, 36.88, 102.6, 82.16, 22.33, 24.17, 240.01]}}


In [2]:
import pandas as pd
import json

# =============================
# 1. 定义映射字典
# =============================
wind_direction_map = {
    '北风': 0,
    '东北风': 1,
    '东风': 2,
    '东南风': 3,
    '南风': 4,
    '西南风': 5,
    '西风': 6,
    '西北风': 7,
    '无持续风向': 8,
    '旋转风': 9
}

wind_power_map = {
    '微风': 0,
    '≤3级': 1,
    '3-4级': 2,
    '4-5级': 3,
    '5-6级': 4
}

time_of_day_map = {'白天': 0, '夜间': 1}
reverse_wind_direction_map = {v: k for k, v in wind_direction_map.items()}
reverse_wind_power_map = {v: k for k, v in wind_power_map.items()}

# =============================
# 2. 读取并清洗数据
# =============================
df = pd.read_csv('end.csv')

# 填充缺失值（可选）
df.fillna({
    '气温': 0,
    '风向编码': 8,   # 默认“无持续风向”
    '风力编码': 0    # 默认“微风”
}, inplace=True)

# =============================
# 3. 添加季节列
# =============================
def get_season(date_str):
    try:
        month = int(date_str.split('/')[1])
    except Exception:
        return '未知'
    if month in [12, 1, 2]:
        return '冬'
    elif month in [3, 4, 5]:
        return '春'
    elif month in [6, 7, 8]:
        return '夏'
    elif month in [9, 10, 11]:
        return '秋'
    else:
        return '未知'

df['season'] = df['日期'].apply(get_season)

# =============================
# 4. 添加温度区间标签
# =============================
def temp_bin(temp):
    if temp > 30:
        return 'above30'
    elif 20 < temp <= 30:
        return '30-20'
    elif 10 < temp <= 20:
        return '20-10'
    elif 0 <= temp <= 10:
        return '10-0'
    else:
        return 'below0'

df['temp_range'] = df['气温'].apply(temp_bin)

# =============================
# 5. 将编码转为中文描述
# =============================
df['wind_direction'] = df['风向编码'].map(reverse_wind_direction_map)
df['wind_force'] = df['风力编码'].map(reverse_wind_power_map)
df['time_of_day'] = df['时间段'].map({0: '夜间', 1: '白天'})

# =============================
# 6. 构建 windSeasonData 结构
# =============================
wind_season_data = {'day': {}, 'night': {}}

# 过滤掉无效数据
df_valid = df[df['wind_direction'].notna() & df['wind_force'].notna()]

# 新增一列，用于分组：将“白天/夜间”转为“day/night”
df_valid['time_of_day_code'] = df_valid['time_of_day'].map({'白天': 'day', '夜间': 'night'})

# 初始化所有温度区间为空数组（避免空值导致JSON异常）
temp_ranges = ['below0', '10-0', '20-10', '30-20', 'above30']
for period in ['day', 'night']:
    for tr in temp_ranges:
        wind_season_data[period][tr] = []

# 分组统计
for time_desc, time_group in df_valid.groupby('time_of_day_code'):
    for temp_range, group in time_group.groupby('temp_range'):
        stats = group.groupby(['wind_direction', 'wind_force', 'season']).size().reset_index(name='count')
        records = [
            {
                "direction": r['wind_direction'],
                "force": r['wind_force'],
                "count": int(r['count']),
                "season": r['season']
            } for _, r in stats.iterrows()
        ]
        wind_season_data[time_desc][temp_range] = records

# =============================
# 7. 输出结果为 JSON
# =============================
output_json = wind_season_data  # 已经是所需结构

# 打印或保存
print(json.dumps(output_json, ensure_ascii=False, indent=2))

# 可选：保存为文件
with open('wind_season_data.json', 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2)

{
  "day": {
    "below0": [
      {
        "direction": "东北风",
        "force": "≤3级",
        "count": 51,
        "season": "冬"
      },
      {
        "direction": "东风",
        "force": "≤3级",
        "count": 195,
        "season": "冬"
      },
      {
        "direction": "东风",
        "force": "≤3级",
        "count": 13,
        "season": "秋"
      },
      {
        "direction": "北风",
        "force": "3-4级",
        "count": 36,
        "season": "冬"
      },
      {
        "direction": "北风",
        "force": "≤3级",
        "count": 50,
        "season": "冬"
      },
      {
        "direction": "南风",
        "force": "≤3级",
        "count": 38,
        "season": "冬"
      },
      {
        "direction": "旋转风",
        "force": "微风",
        "count": 66,
        "season": "冬"
      },
      {
        "direction": "无持续风向",
        "force": "≤3级",
        "count": 844,
        "season": "冬"
      },
      {
        "direction": "无持续风向",
        "force": "≤3级",
        "count

In [1]:
#查找end_cleaned.csv的缺失值
import pandas as pd
# 读取CSV文件
df = pd.read_csv('end_cleaned.csv')
# 检查每列的缺失值数量
missing_values = df.isnull().sum()
# 输出缺失值统计
print(missing_values)

纬度         0
经度         0
SO2        0
NO2        0
PM10       0
CO         0
O31小时      0
O38小时      0
PM2.5      0
污染程度       0
日期         0
时间段        0
气温         0
风向编码       0
风力编码       0
降水强度       0
天气_多云      0
天气_晴       0
天气_阴       0
天气_雨       0
天气_雪       0
天气_雾霾沙尘    0
dtype: int64
